In [5]:
from typing import Any
from __future__ import annotations

from IPython.display import display, Markdown

import sys
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import ast
import isodate
import pandas as pd
import numpy as np

ROOT: Path = Path.cwd().parent
sys.path.append(str(ROOT))

import src.core.util_examplification as examplification
BASE_DIR = Path("../data/raw")

PALETTE: dict[str, str] = {
    "bg": "#0D0F1A",
    "panel": "#141726",
    "accent1": "#FF4D6D",
    "accent2": "#4CC9F0",
    "accent3": "#F4A261",
    "accent4": "#2EC4B6",
    "text": "#E8EAF0",
    "muted": "#6C7293",
}
datasets: dict[str, list[Any]] = {
    "videos": [],
    "comments": [],
    "channels": [],
}
def printf(text_format):
    display(Markdown(text_format))
    
for csv_file in BASE_DIR.rglob("*.csv"):
    try:
        df = pd.read_csv(csv_file)

        df["source_file"] = csv_file.name
        df["source_path"] = str(csv_file)

        id_dir = next(
            (part for part in csv_file.parts if part.startswith("id_")),
            None
        )
        df["collection_id"] = id_dir

        stem_parts = csv_file.stem.split("_")
        country: str = stem_parts[-1] if len(stem_parts) > 2 else None
        df["country"] = country

        filename = csv_file.name.lower()

        if "video" in filename:
            datasets["videos"].append(df)

        elif "comment" in filename:
            datasets["comments"].append(df)

        elif "channel" in filename:
            datasets["channels"].append(df)

    except Exception as e:
        print(f"Erro ao ler {csv_file}: {e}")

videos_df = pd.concat(datasets["videos"], ignore_index=True) \
    if datasets["videos"] else pd.DataFrame()

comments_df = pd.concat(datasets["comments"], ignore_index=True) \
    if datasets["comments"] else pd.DataFrame()

channels_df = pd.concat(datasets["channels"], ignore_index=True) \
    if datasets["channels"] else pd.DataFrame()

print(f"Videos: {len(videos_df)}")
print(f"Comments: {len(comments_df)}")
print(f"Channels: {len(channels_df)}")


Videos: 3597
Comments: 157467
Channels: 3597


In [10]:
df: pd.DataFrame = videos_df.drop_duplicates(subset=["video_id"])
df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")

for col in ["view_count", "like_count", "comment_count"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["video_id", "title", "channel_title", "channel_id", "thumbnail",
            "tags", "duration", "source_file", "source_path", "collection_id", "country"]:
    df[col] = df[col].astype("string").str.strip()
def duration_to_seconds(x):
    if pd.isna(x):
        return np.nan
    return int(isodate.parse_duration(x).total_seconds())

df["duration_seconds"] = df["duration"].apply(duration_to_seconds)
df["is_short"] = df["duration_seconds"] <= 60
def parse_tags(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return [t.strip().lower() for t in ast.literal_eval(x)]
    except:
        return [t.strip().lower() for t in str(x).strip("[]").split(",") if t.strip()]

df["tags_list"] = df["tags"].apply(parse_tags)
df["tags_n"] = df["tags_list"].apply(len)
df["title_len"] = df["title"].fillna("").str.len()
df["title_words"] = df["title"].fillna("").str.split().apply(len)

df["log_view_count"] = np.log1p(df["view_count"])
df["log_like_count"] = np.log1p(df["like_count"])
df["log_comment_count"] = np.log1p(df["comment_count"])
df["log_duration_seconds"] = np.log1p(df["duration_seconds"])
LIKE_WEIGHT = 2
COMMENT_WEIGHT = 5

df_long = df[df["duration_seconds"] >= 240]

df_long["engagement_rate"] = np.where(
    df_long["view_count"].gt(0),
    (
        df_long["like_count"]* LIKE_WEIGHT +
        df_long["comment_count"]* COMMENT_WEIGHT
    ) / df_long["view_count"],
    np.nan
)
df_long["likes_per_view"] = np.where(
    df_long["view_count"] > 0,
    df_long["like_count"] / df_long["view_count"],
    np.nan
)

df_long["comments_per_view"] = np.where(
    df_long["view_count"] > 0,
    df_long["comment_count"] / df_long["view_count"],
    np.nan
)
df = df_long.copy()
df["published_date"] = df["published_at"].dt.date
df["published_year"] = df["published_at"].dt.year
df["published_month"] = df["published_at"].dt.month
df["published_hour"] = df["published_at"].dt.hour
df["published_dayofweek"] = df["published_at"].dt.day_name()

USELESS_FIELDS: list[str] = [
    "thumbnail",
    "source_file",
    "source_path",
    "collection_id",
    "tags",
    "duration",
    "published_at"
]
for field in USELESS_FIELDS:
    
    try:
        df = df.drop(columns=[field])
    except:
        print(f"Campo {field} não existe")

In [11]:
df.dtypes

video_id                 string
title                    string
channel_title            string
channel_id               string
category_id               int64
view_count                int64
like_count              float64
comment_count           float64
country                  string
duration_seconds          int64
is_short                   bool
tags_list                object
tags_n                    int64
title_len                 Int64
title_words               int64
log_view_count          float64
log_like_count          float64
log_comment_count       float64
log_duration_seconds    float64
engagement_rate         float64
likes_per_view          float64
comments_per_view       float64
published_date           object
published_year            int32
published_month           int32
published_hour            int32
published_dayofweek         str
dtype: object